In [1]:
import pandas as pd
import numpy as np
import os

from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from lightgbm.callback import record_evaluation

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import MultipleLocator

import time

import joblib

# Data Preprocessing


In [ ]:
bridge_destroy_ETE = pd.read_csv('/content/bridge_destroy_0_to_99999.csv')

# Removal of the best- and worst-case scenarios
# Scenarios in which the damage states of all bridges are uniformly 0 or 1
bridge_destroy_ETE = bridge_destroy_ETE.drop(0)
bridge_destroy_ETE = bridge_destroy_ETE.drop(1)

bridge_destroy_ETE.reset_index(drop=True, inplace=True)

In [ ]:
# Eliminate abnormal termination scenarios
bridge_destroy_ETE = bridge_destroy_ETE[bridge_destroy_ETE['Abnormal'] == 'P']
bridge_destroy_ETE = bridge_destroy_ETE.reset_index(drop=True)

print(len(bridge_destroy_ETE))

In [ ]:
bridge_cols = [str(i) for i in range(1, 49)]

X = bridge_destroy_ETE[bridge_cols].values

y = bridge_destroy_ETE['ETE'].values

print(f'Number of X samples: {len(X)}')
print(f'Example X sample: {X[0]}')
print(f'Number of y samples: {len(y)}')
print(f'Example y value: {y[0]}')

In [6]:
data_count = 50000

X = X[:data_count]
y = y[:data_count]

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=2/3, random_state=42
)

print(f'X_train : {len(X_train)}')
print(f'X_val : {len(X_val)}')
print(f'X_test : {len(X_test)}')
print(f'y_train : {len(y_train)}')
print(f'y_val : {len(y_val)}')
print(f'y_test : {len(y_test)}')

# Model Training


In [22]:
lgbm_regressor = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=16,
    num_leaves=32,
    min_data_in_leaf=10,
    random_state=42
)

In [ ]:
evals_result = {}

start_time = time.time()

# Train the model using the training dataset
# Use the validation dataset for early stopping and optimal iteration selection
lgbm_regressor.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=['train', 'val'],
    eval_metric='rmse',
    callbacks=[
        early_stopping(stopping_rounds=300),
        log_evaluation(period=10),
        record_evaluation(evals_result)
    ]
)

end_time = time.time()

print(f"Training time: {end_time - start_time:.4f} seconds")
print(f"Best iteration: {lgbm_regressor.best_iteration_}")

# Model Training Results


In [ ]:
y_train_pred = lgbm_regressor.predict(X_train)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print(f"[Train] RMSE : {train_rmse:.3f}")
print(f"[Train] MAE  : {train_mae:.3f}")
print(f"[Train] R²  : {train_r2:.3f}")

In [ ]:
y_val_pred = lgbm_regressor.predict(X_val)

val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
val_mae = mean_absolute_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

print(f"[Validation] RMSE : {val_rmse:.3f}")
print(f"[Validation] MAE  : {val_mae:.3f}")
print(f"[Validation] R²  : {val_r2:.3f}")

In [ ]:
y_test_pred = lgbm_regressor.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"[Test] RMSE : {test_rmse:.3f}")
print(f"[Test] MAE  : {test_mae:.3f}")
print(f"[Test] R²  : {test_r2:.3f}")

# Prediction Time Measurement




In [ ]:
user_input = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]

user_input_array = np.array(user_input).reshape(1, -1)

start_time = time.time()
prediction = lgbm_regressor.predict(user_input_array)
end_time = time.time()

print(f"Prediction result: {prediction[0]:.3f}")
print(f"Prediction time: {end_time - start_time:.4f} seconds")

# Model Saving

In [ ]:
joblib.dump(
    lgbm_regressor,
    f'/content/LightGBM_BestR2_{data_count}.pkl'
)

print("Model saved successfully")